# ASALA-QA Dataset Specifications

This notebook prints a comprehensive descriptive profile of ASALA-QA.

Expected columns:
- `Text`
- `Dialect`
- `Country`

Update `FILE_PATH` in the first code cell to point to your ASALA-QA CSV file.


In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path


FILE_PATH = "/content/drive/MyDrive/ASALA/ASALA_QA.csv"

df = pd.read_csv(FILE_PATH)


df.columns = df.columns.str.strip()

required_columns = ["text", "dialect", "country"]
missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(df.columns)}"
    )

print("ASALA-QA loaded successfully.")
print(f"Shape: {df.shape}")


ASALA-QA loaded successfully.
Shape: (74938, 3)


## 1. Basic Dataset Information


In [4]:
print("=" * 70)
print("ASALA-QA BASIC SPECIFICATIONS")
print("=" * 70)

print(f"Total records: {len(df):,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Columns: {', '.join(df.columns)}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):,.2f} MB")

print("\nData types:")
print(df.dtypes.to_string())


ASALA-QA BASIC SPECIFICATIONS
Total records: 74,938
Number of columns: 3
Columns: text, dialect, country
Exact duplicate rows: 0
Memory usage: 23.40 MB

Data types:
text       object
dialect    object
country    object


## 2. Missing and Empty Values


In [5]:
print("=" * 70)
print("MISSING / EMPTY VALUES")
print("=" * 70)

for col in required_columns:
    missing = df[col].isna().sum()
    empty = df[col].fillna("").astype(str).str.strip().eq("").sum()
    print(
        f"{col:10s} | Missing: {missing:,} "
        f"| Missing/empty combined: {empty:,} "
        f"| Complete: {len(df) - empty:,}"
    )


MISSING / EMPTY VALUES
text       | Missing: 0 | Missing/empty combined: 0 | Complete: 74,938
dialect    | Missing: 0 | Missing/empty combined: 0 | Complete: 74,938
country    | Missing: 0 | Missing/empty combined: 0 | Complete: 74,938


## 3. Dialect Distribution


In [9]:
dialect_counts = df["dialect"].value_counts(dropna=False)
dialect_pct = df["dialect"].value_counts(dropna=False, normalize=True).mul(100)

dialect_stats = pd.DataFrame({
    "Records": dialect_counts,
    "Percentage": dialect_pct.round(2)
})

print("=" * 70)
print("DIALECT DISTRIBUTION")
print("=" * 70)
print(dialect_stats.to_string())
print(f"\nNumber of dialect categories: {df['dialect'].nunique(dropna=True):,}")


DIALECT DISTRIBUTION
         Records  Percentage
dialect                     
EGY        12513       16.70
GLF        12235       16.33
JRD         8034       10.72
SUD         7782       10.38
LYB         6773        9.04
ALG         6400        8.54
IRQ         6397        8.54
LEV         5246        7.00
MGH         5127        6.84
AEB         4431        5.91

Number of dialect categories: 10


## 4. Country Distribution


In [10]:
country_counts = df["country"].value_counts(dropna=False)
country_pct = df["country"].value_counts(dropna=False, normalize=True).mul(100)

country_stats = pd.DataFrame({
    "Records": country_counts,
    "Percentage": country_pct.round(2)
})

print("=" * 70)
print("COUNTRY DISTRIBUTION")
print("=" * 70)
print(country_stats.to_string())
print(f"\nNumber of countries: {df['country'].nunique(dropna=True):,}")


COUNTRY DISTRIBUTION
                  Records  Percentage
country                              
Egypt               12513       16.70
Saudi Arabia        12235       16.33
Jordan               8034       10.72
Sudan                7782       10.38
Libya                6773        9.04
Algeria              6400        8.54
Iraq                 6397        8.54
Levantine Region     5246        7.00
Morocco              5127        6.84
Tunisia              4431        5.91

Number of countries: 10


## 5. Country–Dialect Mapping


In [13]:
country_dialect = pd.crosstab(df["country"], df["dialect"], margins=True)

print("=" * 70)
print("COUNTRY × DIALECT CROSS-TABULATION")
print("=" * 70)
print(country_dialect.to_string())

print("\nObserved country–dialect pairs:")
pairs = (
    df[["country", "dialect"]]
    .dropna()
    .value_counts()
    .reset_index(name="Records")
    .sort_values(["country", "Records"], ascending=[True, False])
)
print(pairs.to_string(index=False))


COUNTRY × DIALECT CROSS-TABULATION
dialect            AEB   ALG    EGY    GLF   IRQ   JRD   LEV   LYB   MGH   SUD    All
country                                                                              
Algeria              0  6400      0      0     0     0     0     0     0     0   6400
Egypt                0     0  12513      0     0     0     0     0     0     0  12513
Iraq                 0     0      0      0  6397     0     0     0     0     0   6397
Jordan               0     0      0      0     0  8034     0     0     0     0   8034
Levantine Region     0     0      0      0     0     0  5246     0     0     0   5246
Libya                0     0      0      0     0     0     0  6773     0     0   6773
Morocco              0     0      0      0     0     0     0     0  5127     0   5127
Saudi Arabia         0     0      0  12235     0     0     0     0     0     0  12235
Sudan                0     0      0      0     0     0     0     0     0  7782   7782
Tunisia           4

## 6. Text Length Statistics


In [14]:
text = df["text"].fillna("").astype(str)

df_stats = pd.DataFrame(index=df.index)
df_stats["word_count"] = text.str.split().str.len()
df_stats["character_count"] = text.str.len()

print("=" * 70)
print("TEXT LENGTH STATISTICS")
print("=" * 70)

print("\nWord count:")
print(df_stats["word_count"].describe(percentiles=[.25, .5, .75, .90, .95, .99]).round(2))

print("\nCharacter count:")
print(df_stats["character_count"].describe(
    percentiles=[.25, .5, .75, .90, .95, .99]
).round(2))

print(f"\nShortest record (words): {df_stats['word_count'].min():,}")
print(f"Longest record (words): {df_stats['word_count'].max():,}")
print(f"Mean words per record: {df_stats['word_count'].mean():.2f}")
print(f"Median words per record: {df_stats['word_count'].median():.2f}")


TEXT LENGTH STATISTICS

Word count:
count    74938.00
mean         8.08
std          0.44
min          1.00
25%          8.00
50%          8.00
75%          8.00
90%          8.00
95%          8.00
99%         10.00
max         17.00
Name: word_count, dtype: float64

Character count:
count    74938.00
mean        39.80
std          5.58
min          6.00
25%         36.00
50%         39.00
75%         43.00
90%         47.00
95%         49.00
99%         53.00
max        227.00
Name: character_count, dtype: float64

Shortest record (words): 1
Longest record (words): 17
Mean words per record: 8.08
Median words per record: 8.00


## 7. Text Length by Dialect


In [16]:
length_by_dialect = (
    pd.DataFrame({
        "Dialect": df["dialect"],
        "word_count": df_stats["word_count"],
        "character_count": df_stats["character_count"]
    })
    .groupby("Dialect")
    .agg(
        Records=("word_count", "size"),
        Total_Words=("word_count", "sum"),
        Mean_Words=("word_count", "mean"),
        Median_Words=("word_count", "median"),
        Min_Words=("word_count", "min"),
        Max_Words=("word_count", "max"),
        Mean_Characters=("character_count", "mean")
    )
    .round(2)
)

print(length_by_dialect.to_string())


         Records  Total_Words  Mean_Words  Median_Words  Min_Words  Max_Words  Mean_Characters
Dialect                                                                                       
AEB         4431        35448        8.00           8.0          8          8            40.55
ALG         6400        51200        8.00           8.0          8          8            41.30
EGY        12513       100104        8.00           8.0          8          8            36.92
GLF        12235        97880        8.00           8.0          8          8            39.57
IRQ         6397        51176        8.00           8.0          8          8            39.97
JRD         8034        64272        8.00           8.0          8          8            40.47
LEV         5246        41968        8.00           8.0          8          8            39.10
LYB         6773        54184        8.00           8.0          8          8            39.27
MGH         5127        47006        9.17         

## 8. Vocabulary and Lexical Diversity by Dialect


In [18]:
def lexical_statistics(series):
    tokens = []
    for value in series.dropna().astype(str):
        tokens.extend(value.split())

    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    ttr = unique_tokens / total_tokens if total_tokens else 0.0

    if total_tokens:
        counts = pd.Series(tokens).value_counts().to_numpy(dtype=float)
        probs = counts / counts.sum()
        entropy = -(probs * np.log2(probs)).sum()
    else:
        entropy = 0.0

    return pd.Series({
        "Total Tokens": total_tokens,
        "Unique Tokens": unique_tokens,
        "TTR": ttr,
        "Shannon Entropy": entropy
    })

lexical_by_dialect = (
    df.groupby("dialect")["text"]
      .apply(lexical_statistics)
      .unstack()
)

lexical_by_dialect["Total Tokens"] = lexical_by_dialect["Total Tokens"].astype(int)
lexical_by_dialect["Unique Tokens"] = lexical_by_dialect["Unique Tokens"].astype(int)
lexical_by_dialect["TTR"] = lexical_by_dialect["TTR"].round(6)
lexical_by_dialect["Shannon Entropy"] = lexical_by_dialect["Shannon Entropy"].round(5)

print("=" * 70)
print("LEXICAL STATISTICS BY DIALECT")
print("=" * 70)
print(lexical_by_dialect.to_string())
print(
    "\nNote: token statistics use whitespace-based tokenization and therefore "
    "represent surface-form token measurements rather than morphology-aware "
    "Arabic lexical segmentation."
)


LEXICAL STATISTICS BY DIALECT
         Total Tokens  Unique Tokens       TTR  Shannon Entropy
dialect                                                        
AEB             35448          10109  0.285178         11.17299
ALG             51200          13846  0.270430         11.65781
EGY            100104          15781  0.157646         10.37503
GLF             97880          20504  0.209481         11.57807
IRQ             51176          13914  0.271885         11.39277
JRD             64272          14863  0.231252         11.28860
LEV             41968          10863  0.258840         10.95122
LYB             54184          12123  0.223738         10.93389
MGH             47006          13698  0.291410         11.53061
SUD             62256          11337  0.182103         10.48174

Note: token statistics use whitespace-based tokenization and therefore represent surface-form token measurements rather than morphology-aware Arabic lexical segmentation.


## 9. Exact Duplicate Analysis


In [20]:
ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
)

def normalize_arabic_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    value = ARABIC_DIACRITICS.sub("", value)
    value = value.replace("\u0640", "")  # tatweel
    value = re.sub(r"\s+", " ", value).strip()
    return value

normalized_text = df["text"].apply(normalize_arabic_text)

counts = normalized_text.value_counts()
repeated = counts[counts > 1]

total_records = len(df)
unique_normalized_texts = normalized_text.nunique()
records_in_duplicate_groups = int(repeated.sum())
distinct_duplicated_texts = len(repeated)
redundant_occurrences = int((repeated - 1).sum())
uniqueness_rate = unique_normalized_texts / total_records if total_records else 0

print("=" * 70)
print("EXACT NORMALIZED-TEXT DUPLICATE ANALYSIS")
print("=" * 70)
print(f"Total records: {total_records:,}")
print(f"Unique normalized texts: {unique_normalized_texts:,}")
print(f"Text-level uniqueness rate: {uniqueness_rate:.2%}")
print(f"Distinct texts with repeated occurrences: {distinct_duplicated_texts:,}")
print(f"Records involved in duplicate groups: {records_in_duplicate_groups:,}")
print(f"Redundant occurrences beyond the first: {redundant_occurrences:,}")

print(
    "\nNote: this is exact matching after text normalization, "
    "not semantic near-duplicate detection."
)


EXACT NORMALIZED-TEXT DUPLICATE ANALYSIS
Total records: 74,938
Unique normalized texts: 74,938
Text-level uniqueness rate: 100.00%
Distinct texts with repeated occurrences: 0
Records involved in duplicate groups: 0
Redundant occurrences beyond the first: 0

Note: this is exact matching after text normalization, not semantic near-duplicate detection.


## 10. Cross-Dialect Exact Text Overlap


In [22]:
temp = pd.DataFrame({
    "normalized_text": normalized_text,
    "Dialect": df["dialect"]
})

valid = temp[
    temp["normalized_text"].ne("") &
    temp["Dialect"].notna()
]

cross = (
    valid.groupby("normalized_text")["Dialect"]
    .agg(
        dialect_count="nunique",
        dialects=lambda x: "|".join(sorted(set(map(str, x)))),
        occurrences="size"
    )
    .reset_index()
)

cross = cross[cross["dialect_count"] > 1].copy()
cross["word_count"] = cross["normalized_text"].str.split().str.len()

print("=" * 70)
print("CROSS-DIALECT EXACT OVERLAP")
print("=" * 70)
print(f"Distinct texts appearing under multiple dialect labels: {len(cross):,}")

if len(cross):
    print("\nMost frequent dialect-label combinations:")
    print(cross["dialects"].value_counts().head(20).to_string())


CROSS-DIALECT EXACT OVERLAP
Distinct texts appearing under multiple dialect labels: 0


## 11. Arabic / Latin Character Profile


In [23]:
latin_mask = text.str.contains(r"[A-Za-z]", regex=True, na=False)
arabic_mask = text.str.contains(r"[\u0600-\u06FF]", regex=True, na=False)

print("=" * 70)
print("SCRIPT PROFILE")
print("=" * 70)
print(f"Records containing Arabic characters: {arabic_mask.sum():,} ({arabic_mask.mean():.2%})")
print(f"Records containing Latin characters: {latin_mask.sum():,} ({latin_mask.mean():.2%})")
print(f"Records containing both Arabic and Latin characters: {(arabic_mask & latin_mask).sum():,}")


SCRIPT PROFILE
Records containing Arabic characters: 74,920 (99.98%)
Records containing Latin characters: 177 (0.24%)
Records containing both Arabic and Latin characters: 159


## 12. Complete Summary


In [25]:
print("=" * 70)
print("ASALA-QA SUMMARY")
print("=" * 70)

print(f"Records                 : {len(df):,}")
print(f"Columns                 : {', '.join(required_columns)}")
print(f"Dialect categories      : {df['dialect'].nunique(dropna=True):,}")
print(f"Countries               : {df['country'].nunique(dropna=True):,}")
print(f"Total whitespace tokens : {int(df_stats['word_count'].sum()):,}")
print(f"Mean words/record       : {df_stats['word_count'].mean():.2f}")
print(f"Median words/record     : {df_stats['word_count'].median():.2f}")
print(f"Unique normalized texts : {unique_normalized_texts:,}")
print(f"Uniqueness rate         : {uniqueness_rate:.2%}")
print(f"Cross-dialect overlaps  : {len(cross):,}")

print("\nMissing/empty values:")
for col in required_columns:
    empty = df[col].fillna("").astype(str).str.strip().eq("").sum()
    print(f"  {col:8s}: {empty:,}")


ASALA-QA SUMMARY
Records                 : 74,938
Columns                 : text, dialect, country
Dialect categories      : 10
Countries               : 10
Total whitespace tokens : 605,494
Mean words/record       : 8.08
Median words/record     : 8.00
Unique normalized texts : 74,938
Uniqueness rate         : 100.00%
Cross-dialect overlaps  : 0

Missing/empty values:
  text    : 0
  dialect : 0
  country : 0
